In [4]:
# 1. O truque do SQLite continua sendo necessário neste ambiente sempre que usarmos o Chroma
import sys
__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.get('pysqlite3')

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 2. Precisamos chamar o mesmo "cartógrafo" (modelo de embeddings) 
# para que ele saiba traduzir a pergunta do usuário para o mesmo formato do banco
print("Carregando o modelo de embeddings...")
modelo_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Apontamos para a pasta onde o banco foi salvo
diretorio_banco = "../data/chroma_banco_vetorial"

# 4. Carregamos o banco já pronto do disco (sem usar o 'from_documents')
print("Conectando ao ChromaDB local...")
banco_vetorial = Chroma(
    persist_directory=diretorio_banco,
    embedding_function=modelo_embeddings
)

# Validação: vamos contar quantas fichas existem lá dentro
total_documentos = banco_vetorial._collection.count()
print(f"Banco carregado. Total de fragmentos disponíveis para busca: {total_documentos}")

Carregando o modelo de embeddings...


/home/mariana/Projetinhos pessoais/projeto-extra-favelas/.venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Conectando ao ChromaDB local...
Banco carregado. Total de fragmentos disponíveis para busca: 1553


Testando o ChromaDB

In [7]:
# 1. Transformamos o nosso banco vetorial em um "Buscador" (Retriever)
# O parâmetro k=3 diz para ele trazer apenas as 3 "fichas" mais parecidas com a pergunta
buscador = banco_vetorial.as_retriever(search_kwargs={"k": 3})

# 2. Definimos uma pergunta de teste em linguagem natural
# (Você pode mudar essa pergunta para testar assuntos que sabe que estão no seu CSV)
pergunta = "Ao analisar as matérias sobre favela fora do contexto de grandes eventos e festivais, quais são os principais assuntos abordados?"

# 3. Acionamos o buscador passando a pergunta
print(f"Buscando pela pergunta: '{pergunta}'\n")
documentos_encontrados = buscador.invoke(pergunta)

# 4. Lemos o que o banco nos trouxe de volta
for i, doc in enumerate(documentos_encontrados, 1):
    print(f"--- Fragmento {i} ---")
    # Imprime os primeiros 250 caracteres do texto para a gente ter uma ideia
    print(f"TEXTO: {doc.page_content[:250]}...") 
    # Imprime os metadados (nossa garantia de não alucinação)
    print(f"FONTE: {doc.metadata}")
    print("-" * 40 + "\n")

Buscando pela pergunta: 'Ao analisar as matérias sobre favela fora do contexto de grandes eventos e festivais, quais são os principais assuntos abordados?'

--- Fragmento 1 ---
TEXTO: Segundo Preto Zezé, o evento é um espaço fundamental para desestigmatizar as favelas:

— A minha expectativa é que este encontro ajude a gente a dar visibilidade a outro olhar sobre a favela, que não se resuma só a tragédia e dificuldades. Que ela nã...
FONTE: {'data_publicacao': '2026-10', 'titulo': 'Fórum EXTRA Favelas debaterá futuro das comunidades cariocas e contará com show inédito; saiba como se inscrever', 'url': 'https://extra.globo.com/extra-favelas/noticia/2026/10/forum-extra-favelas-debatera-futuro-das-comunidades-cariocas-e-contara-com-show-inedito-saiba-como-se-inscrever.ghtml'}
----------------------------------------

--- Fragmento 2 ---
TEXTO: Que tipo de projeto pode concorrer

As propostas devem se encaixar em um de quatro segmentos. Em artes cênicas e visuais, entram espetáculos de tea

## Montando a cadeia RAG

In [12]:
import os
import json
import requests
from getpass import getpass

# 1. Configura a chave de API de forma segura
if "GOOGLE_API_KEY" not in os.environ:
    chave = getpass("Insira sua chave da API do Gemini (AIza...): ")
    os.environ["GOOGLE_API_KEY"] = chave

# 2. Função auxiliar para formatar os trechos recuperados do ChromaDB
def formatar_documentos(docs):
    bloco_textos = []
    for doc in docs:
        titulo = doc.metadata.get("titulo", "Sem título")
        data = doc.metadata.get("data_publicacao", doc.metadata.get("data", "Data desconhecida"))
        url = doc.metadata.get("url", "Sem URL")
        
        texto_item = f"--- MATÉRIA: {titulo} ({data}) | Link: {url} ---\n{doc.page_content}\n"
        bloco_textos.append(texto_item)
    return "\n\n".join(bloco_textos)

# 3. Função principal do Pipeline RAG (Usando requisição HTTP direta)
def executar_rag(pergunta):
    # A. O ChromaDB resgata os trechos mais relevantes do seu CSV
    docs_encontrados = buscador.invoke(pergunta)
    contexto_formatado = formatar_documentos(docs_encontrados)
    
    # B. Montagem do Prompt de Grounding Estrito
    prompt_completo = f"""
Você é uma pesquisadora especialista em análise de mídia e sociologia urbana.
Sua tarefa é responder à pergunta do usuário baseando-se EXCLUSIVAMENTE nos trechos de notícias fornecidos abaixo.

Regras estritas:
1. Não invente informações e não use conhecimentos externos que não estejam no contexto fornecido.
2. Se o contexto fornecido não contiver elementos suficientes para responder, diga claramente: "Não encontrei informações suficientes no corpus para responder a essa pergunta."
3. Sempre cite o título, a data de publicação e a URL da matéria ao mencionar qualquer fato, evento ou trecho.

Contexto das matérias recuperadas do Extra:
{contexto_formatado}

Pergunta: {pergunta}

Resposta analítica:
"""
    
    # C. Conexão HTTP direta usando a versão EXATA pedida pela API (3.8-flash)
    chave_api = os.environ["GOOGLE_API_KEY"]
    url_gemini = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent?key={chave_api}"
    
    headers = {'Content-Type': 'application/json'}
    payload = {
        "contents": [{"parts": [{"text": prompt_completo}]}]
    }
    
    # Faz a chamada para a API
    resposta = requests.post(url_gemini, headers=headers, data=json.dumps(payload))
    
    # D. Extraindo a resposta do formato JSON retornado
    if resposta.status_code == 200:
        dados = resposta.json()
        return dados['candidates'][0]['content']['parts'][0]['text']
    else:
        return f"Erro na API: {resposta.status_code} - {resposta.text}"

print("Pipeline RAG configurado com sucesso usando conexão HTTP universal (Gemini 3.8)!")

Pipeline RAG configurado com sucesso usando conexão HTTP universal (Gemini 3.8)!


In [ ]:
pergunta_analitica = (
    "Ao analisar as matérias sobre favela que não tratam do festival Rock in Rio, "
    "quais são os principais temas abordados?"
)

print("Processando a resposta na nuvem...\n")
resultado = executar_rag(pergunta_analitica)
print(resultado)

In [ ]:
pergunta_analitica = (
    "Ao analisar as matérias sobre favela que não tratam do festival Rock in Rio, "
    "qual é o tom predominantemente adotado (negativo ou positivo)?"
)

print("Processando a resposta na nuvem...\n")
resultado = executar_rag(pergunta_analitica)
print(resultado)